In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from sympy import latex

In [ ]:
#Function that integrate a symbolic function ff  multiply by the weight wei on the variable x in (-binf,bsup) using the normalized Lebesgue measure
def esperance(ff,wei, u,bmin,bmax):
  
#    ff : expression sympCy
#        Fonction à intégrer
#    x : symbol sympy        

    return sp.integrate(ff*wei, (u, bmin, bmax))

In [ ]:
#defining the conditional density  of Y
#u is a real input
#the output is the conditional density
def fX(u):
    return sp.Piecewise(
    (1/(2*x1), (u >= -x1) & (u <= x1)),
    (0, True)
)

In [ ]:
#defining the  density  of X depending on a>0
#u is a real input
#the output is the density of X at u
def densX(u,aa):
    return sp.Piecewise(
    (a*u**(aa-1), (u >= 0) & (u <= 1)),
    (0, True)
)

In [ ]:
# Symbolic variables definition
x1, x2,tt,t,s= sp.symbols('x1 x2 tt t s')
#
a = sp.symbols('a', positive=True, real=True)

In [ ]:
a=10;
fxx=fX(tt);
ddens=densX(x1,a);
fy=esperance(fxx,ddens, x1,0,1);
FY=sp.integrate(fxx,(tt,0,t)) 
print(latex(FY))
FY_num = sp.lambdify(t, FY, "numpy");
t_vals = np.arange(0.5, 1.1, 0.1);
#fff = np.vectorize(FY_num)(t_vals);
#fff=FY_num(t_vals);
print(t_vals)
print(fff)
#ff=fy_num(t_vals);
#plt.plot(t_vals,ff);

In [ ]:
FXt = sp.Piecewise(
    (1/(2*x1), (t >= -x1) & (t <= x1)),
    (0, True)
)
FXs = sp.Piecewise(
    (1/(2*x1), (s >= -x1) & (s <= x1)),
    (0, True)
)

In [ ]:
FFXt=a*FXt*x1**(a-1);
FFXs=a*FXs*x1**(a-1);

In [ ]:
print(esperance(FFXt*FFXs,x1,0,1))

In [ ]:
# Computing the conditional expectation of f with respect to X1=x1
esperance_condf=esperance(esperance(f,x3),x2)
# Computing the conditional expectation of f² with respect to X1=x1
esperance_condf2=esperance(esperance(f2,x3),x2)

In [ ]:
#Computing Sigma0
#Building first the vector(esperance_condf)^2 esperance_condf espeance_condf² 
vbric_Sigma0 = sp.Matrix([esperance_condf*esperance_condf, esperance_condf, esperance_condf2])
#Computing the centering part
espvbric_Sigma0=esperance(vbric_Sigma0,x1)
centrageSigma0=espvbric_Sigma0*espvbric_Sigma0.T
#Computing the quadratic part before integrating with respect to x1
mbric_Sigma0=vbric_Sigma0*vbric_Sigma0.T
#Computing the covariance matrix Sigma0
Sigma0=esperance(mbric_Sigma0, x1)-centrageSigma0

In [ ]:
#Computing Sigma_a
vbric_Sigma_a = sp.Matrix([2*esperance_condf, 1, 1])
unzeze=sp.Matrix([1, 0, 0])
matrix_de_1_et_0=unzeze*unzeze.T
covepsf=esperance_condf2-esperance_condf*esperance_condf
Sigma_a=vbric_Sigma_a*vbric_Sigma_a.T+covepsf*matrix_de_1_et_0

In [ ]:
#Computing Sigma_b
vbric_Sigma_b= sp.Matrix([f, f, f2])
#Computing the conditional expectation of the last vector
espvbric_Sigma_b=sp.Matrix([esperance_condf, esperance_condf, esperance_condf2])
#Computing Sigma_b
mbric_Sigma_b=vbric_Sigma_b*vbric_Sigma_b.T
Sigma_b=esperance(esperance(mbric_Sigma_b,x2),x3)-espvbric_Sigma_b*espvbric_Sigma_b.T

In [ ]:
#Computing Sigma1
# Computing the integrand Hadamard product of Sigma_a and Sigma_b
Igrand_Sigma1=sp.hadamard_product(Sigma_a,Sigma_b)
#Finally integrating with respect to x1
Sigma1=esperance(Igrand_Sigma1, x1)

In [ ]:
#Computuation of Sobol' index for X1 and the variance of f
#Expectation of f
espf=esperance(esperance_condf,x1)
#Expectation of f²
espf2=esperance(esperance_condf2,x1)
#Variance of f
varY=espf2-espf*espf
#Computation of the variance of the conditional expectation
numerator_Zobol=esperance(esperance_condf*esperance_condf,x1)-espf*espf
#Computing Sobol' index
Zobol=sp.simplify(numerator_Zobol/varY)
print(latex(sp.simplify(Zobol)))

In [ ]:
#Computing the asymptotique variance of Sobol' index  Chatterjee estimator
v_f=(sp.Matrix([1, 2*espf*(Zobol-1), -Zobol]))/varY
#Asymptotic variance final formula
sigma_Zobol_hat=sp.simplify(v_f.T*(Sigma0+Sigma1)*v_f)
#print(latex(sp.simplify(sigma_Zobol_hat)))

In [ ]:
#Evaluating numerically Sobol' index and the asymptotic variance for special value of a and b
#aa=input('Give a value for a');
#bb=input('Give a value for b');
aa=10.;
bb=2.
values = {a: aa, b: bb}
Zobol_num=Zobol.evalf(subs=values);
sigma_Zobol_hat_num=sigma_Zobol_hat.evalf(subs=values);
print('In the special case a=', aa, 'and b=',bb);
print('Sobol index is: ', Zobol_num);
print('The asymptotic variance is: ',sigma_Zobol_hat_num)

In [ ]:
#Evauating numerically Sobol' index and the asymptotic variance for special value of a and for a serie of b
#Transforming in numerical functions
Zobol_num = sp.lambdify((a, b), Zobol, "numpy")
sigma_Zobol_hat_num = sp.lambdify((a, b), sigma_Zobol_hat, "numpy")
# a = 10
a_val = 10
# b moves from 0 to 2
b_vals = np.arange(0, 2.01, 0.01)
Zobol_vals = Zobol_num(10., b_vals)
sigma_Zobol_hat_vals = sigma_Zobol_hat_num(10, b_vals).reshape(-1)
varestimate=np.loadtxt("var_est_Ishi.dat")
S1=np.loadtxt("S1_est_Chat_Ishi.dat")

In [ ]:
# Create two plot asided
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ----- first plot -----
axes[0].plot(b_vals,Zobol_vals,b_vals,S1)
axes[0].set_title("Sobol'index first variable in Ishigami (a=10)")
axes[0].set_xlabel("b")
axes[0].set_ylabel("Sobol' index and its estimate")
axes[0].grid(True)

# ----- second plot -----
axes[1].plot(b_vals,sigma_Zobol_hat_vals,b_vals,varestimate)
axes[1].set_title("Variance of the Chatterjee estimator (a=10)")
axes[1].set_xlabel("b")
axes[1].set_ylabel("Asymptotic variance and its estimate")
axes[1].grid(True)

# Ajusting spaces automatically
plt.tight_layout()

#saving the plot
plt.savefig("graph_Sob_Ishigami.png", dpi=300)
#Display
plt.show()